# 🏒 Player Tracking Analysis - Zone Entry Method and Offensive Network Structure 
*Data: Big Data Cup 2021 data*


In [1]:
from scipy.spatial import Voronoi, voronoi_plot_2d
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import matplotlib.animation as animation
import seaborn as sns

print("Libraries loaded successfully!")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Libraries loaded successfully!


In [4]:
df = pd.read_csv("hackathon_nwhl.csv")

In [6]:
df.head()

,game_date,Home Team,Away Team,Period,Clock,Home Team Skaters,Away Team Skaters,Home Team Goals,Away Team Goals,Team,...,Event,X Coordinate,Y Coordinate,Detail 1,Detail 2,Detail 3,Detail 4,Player 2,X Coordinate 2,Y Coordinate 2
0,2021-01-23,Minnesota Whitecaps,Boston Pride,1,20:00,5,5,0,0,Boston Pride,...,Faceoff Win,100,43,Backhand,NaN,NaN,NaN,Stephanie Anderson,NaN,NaN
1,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:58,5,5,0,0,Boston Pride,...,Puck Recovery,107,40,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:57,5,5,0,0,Boston Pride,...,Zone Entry,125,28,Carried,NaN,NaN,NaN,Maddie Rowe,NaN,NaN
3,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:55,5,5,0,0,Boston Pride,...,Shot,131,28,Snapshot,On Net,t,f,NaN,NaN,NaN
4,2021-01-23,Minnesota Whitecaps,Boston Pride,1,19:53,5,5,0,0,Boston Pride,...,Faceoff Win,169,21,Backhand,NaN,NaN,NaN,Stephanie Anderson,NaN,NaN


In [20]:
df.columns.tolist()

['game_date',
 'Home Team',
 'Away Team',
 'Period',
 'Clock',
 'Home Team Skaters',
 'Away Team Skaters',
 'Home Team Goals',
 'Away Team Goals',
 'Team',
 'Player',
 'Event',
 'X Coordinate',
 'Y Coordinate',
 'Detail 1',
 'Detail 2',
 'Detail 3',
 'Detail 4',
 'Player 2',
 'X Coordinate 2',
 'Y Coordinate 2']

In [26]:
df['Event'].value_counts()

Event
Puck Recovery      8214
Play               7249
Incomplete Play    3430
Zone Entry         1944
Shot               1909
Dump In/Out        1863
Takeaway           1207
Faceoff Win         846
Penalty Taken       144
Goal                 76
Name: count, dtype: int64

In [29]:
df["Detail 1"].value_counts()

Detail 1
Direct                       6720
Indirect                     3959
Lost                         1637
Carried                      1267
Snapshot                      888
Wristshot                     718
Backhand                      618
Dumped                        578
Forehand                      227
Slapshot                      227
Retained                      226
Played                         99
Fan                            68
Deflection                     53
Tripping                       34
Wrap Around                    31
Interference                   29
Hooking                        21
Roughing                       19
Holding                        11
Cross-checking                  7
Slashing                        6
Too many men on the ice         4
Illegal Check to the Head       2
Game Misconduct                 2
High-sticking                   2
Unsportsmanlike conduct         2
Charging                        1
Feet                            1
Elbow

We want our data to be split into each period, so that when we are identifying networks each network stays inside its our period of play.

In [39]:
period_one_events = df[df['Period'] == 1]
period_two_events = df[df['Period'] == 2]
period_three_events = df[df['Period'] == 3]